---
# **Crime data Exploratory Data Analysis: Pre-processing**
---

# **Introduction to the Dataset**

This dataset contains crime information recorded by police forces across various location in the UK. The goal in this notebook is to clean the data.

Columns Overview:

- **`Crime ID`** – Unique identifier assigned to each recorded crime incident
- **`Month`** – The month when the crime was recorded

- **`Reported by`** – The police force that recorded the crime

- **`Falls within`** – The police force jurisdiction responsible for the location where the crime occured

- **`Longitude`** – Geographic coordinate indicating the east-west position of the crime location

- **`Latitude`** – Geographic coordinate indicating the north-south position of the crime location

- **`Location`** – Approximate street or area where the crime occured

- **`LSOA code`** – Code representing the Lower Layer Super Output Area (a small geographical statistical area used in the UK)

- **`LSOA name`** – name of LSOA

- **`Crime type`** – Category describing the type of crime

- **`Last outcome category`** – The most recent known outcome of the police investigation related to the crime

- **`Context`** – Additional contextual information about the crime event


# **Contents**

The objective for the pre-processing is to clean and profile a datasets for one policeforce, then automate or create a repeatable preprocessing pipleine. The content of this notebook is as followed:

1. Import Libraries
2. Inspect City of London
3. Standardise column names
4. Handle missing values
4. Convert correct datatypes
5. Create time features

7. Remove duplicates
8. Validate relationships
9. Load all CSV files
10. Automate cleaning process
11. Final validation checks
12. Export cleaned dataset as CSV file


# **1. Import Libraries**


In [59]:
import pandas as pd # to work with dataframes
import glob # to import multiple files at once

# **2. Inspect City of London**

In [ ]:
# import city of london cdv files
city_files = sorted(glob.glob("../data/*/*city-of-london-street.csv")) # glob.glob returns a list of file paths matching the pattern, sorted to maintain order
city = pd.concat([pd.read_csv(file) for file in city_files], ignore_index=True) # ignore_index to reset the index after concatenation

In [3]:
city.columns # check the columns of the dataframe to understand the data and identify any issues with the data

Index(['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude',
       'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type',
       'Last outcome category', 'Context'],
      dtype='object')

In [4]:
city.head() # check the first 5 rows of the dataframe to understand the data

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01,City of London Police,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified,NaN
1,e60a5ac62a80e866453254474137c3206417422c62f0c0...,2024-01,City of London Police,City of London Police,-0.107682,51.517786,On or near B521,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,NaN
2,986f618142ec52b7f254e4b0549da2f17ceeb0e130db6c...,2024-01,City of London Police,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Other theft,Investigation complete; no suspect identified,NaN
3,05dc27a88748356f6d59b0bd1389710ebfb42b37e565af...,2024-01,City of London Police,City of London Police,-0.111596,51.518281,On or near Chancery Lane,E01000914,Camden 028B,Theft from the person,Status update unavailable,NaN
4,373d78e2ccec5d05a547cd4bee19045a9e050042a0e6e7...,2024-01,City of London Police,City of London Police,-0.112096,51.515942,On or near Nightclub,E01000914,Camden 028B,Theft from the person,Investigation complete; no suspect identified,NaN


In [5]:
city.info() # check the info of the dataframe to understand the data types and null values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19577 entries, 0 to 19576
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               19042 non-null  object 
 1   Month                  19577 non-null  object 
 2   Reported by            19577 non-null  object 
 3   Falls within           19577 non-null  object 
 4   Longitude              17937 non-null  float64
 5   Latitude               17937 non-null  float64
 6   Location               19577 non-null  object 
 7   LSOA code              17937 non-null  object 
 8   LSOA name              17937 non-null  object 
 9   Crime type             19577 non-null  object 
 10  Last outcome category  19042 non-null  object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 1.8+ MB


# **3. Standardise Column Names**

In [6]:
city.columns = (
    city.columns
    .str.strip() # remove any leading or trailing whitespace from the column names
    .str.lower() # convert the column names to lowercase
    .str.replace(" ", "_") # replace any spaces in the column names with underscores   
)
city.columns # check the column names to ensure they have been cleaned properly

Index(['crime_id', 'month', 'reported_by', 'falls_within', 'longitude',
       'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type',
       'last_outcome_category', 'context'],
      dtype='object')

# **4. Check for Null Values**

In [7]:
display(city.isnull().sum(), round(city.isnull().mean(), 2)) # check the number of null values in each column and the percentage of null values in each column

crime_id                   535
month                        0
reported_by                  0
falls_within                 0
longitude                 1640
latitude                  1640
location                     0
lsoa_code                 1640
lsoa_name                 1640
crime_type                   0
last_outcome_category      535
context                  19577
dtype: int64

crime_id                 0.03
month                    0.00
reported_by              0.00
falls_within             0.00
longitude                0.08
latitude                 0.08
location                 0.00
lsoa_code                0.08
lsoa_name                0.08
crime_type               0.00
last_outcome_category    0.03
context                  1.00
dtype: float64

In [8]:
# remove context column as the column is completely empty and has no values
city = city.drop(columns=["context"], inplace=False)
city.columns # check if the column has been removed

Index(['crime_id', 'month', 'reported_by', 'falls_within', 'longitude',
       'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type',
       'last_outcome_category'],
      dtype='object')

In [9]:
# show rows with null values on longitude
city[city["longitude"].isnull()] # check the rows with null values in longitude to

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category
657,NaN,2024-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Anti-social behaviour,NaN
658,NaN,2024-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Anti-social behaviour,NaN
659,b6ce3aff678d3d356e3200bd9e861a90326b3ed55abded...,2024-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Bicycle theft,Investigation complete; no suspect identified
660,c1c219c13e2833b1c591c3d9749f0945af33abc460d184...,2024-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Bicycle theft,Investigation complete; no suspect identified
661,c939b3c277566f5954e3aa82b1beadce3f3db7d2403d47...,2024-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Burglary,Unable to prosecute suspect
...,...,...,...,...,...,...,...,...,...,...,...
19572,dc01f91df11c080deb8ac6bce6ef0cd769150a64391446...,2026-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation
19573,08c94fe2b9a5edcd6fe252a24ce72733fe1ba542bb62bf...,2026-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation
19574,fb492426195295667d8dba3703ccb013bfee7198892347...,2026-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation
19575,7303457bf2bf9564eb1a610706e87643d52e0666d9ea7d...,2026-01,City of London Police,City of London Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Unable to prosecute suspect


**Dropping rows with no location data, as it is crucial for map visualisation and evaluation of real estate.**

In [10]:
# drop rows with no location data
city = city.dropna(subset=["longitude"], inplace=False)
round(city.isnull().mean(), 2) # check the number of null values in each column to ensure the rows with missing location data have been removed

crime_id                 0.03
month                    0.00
reported_by              0.00
falls_within             0.00
longitude                0.00
latitude                 0.00
location                 0.00
lsoa_code                0.00
lsoa_name                0.00
crime_type               0.00
last_outcome_category    0.03
dtype: float64

In [11]:
# rows with null values in crime_id
city[city["crime_id"].isnull()] # check the rows with null values in crime_id to understand if there are any patterns in the missing data

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category
10,NaN,2024-01,City of London Police,City of London Police,-0.097078,51.519045,On or near A1,E01000001,City of London 001A,Anti-social behaviour,NaN
11,NaN,2024-01,City of London Police,City of London Police,-0.097103,51.518461,On or near Aldersgate Street,E01000001,City of London 001A,Anti-social behaviour,NaN
78,NaN,2024-01,City of London Police,City of London Police,-0.077511,51.515812,On or near Artizan Street,E01000005,City of London 001E,Anti-social behaviour,NaN
265,NaN,2024-01,City of London Police,City of London Police,-0.081113,51.516869,On or near Bishopsgate,E01032739,City of London 001F,Anti-social behaviour,NaN
266,NaN,2024-01,City of London Police,City of London Police,-0.088785,51.517102,On or near Moorgate,E01032739,City of London 001F,Anti-social behaviour,NaN
...,...,...,...,...,...,...,...,...,...,...,...
19013,NaN,2026-01,City of London Police,City of London Police,-0.087157,51.512624,On or near Lombard Street,E01032739,City of London 001F,Anti-social behaviour,NaN
19014,NaN,2026-01,City of London Police,City of London Police,-0.095764,51.512549,On or near Friday Street,E01032739,City of London 001F,Anti-social behaviour,NaN
19015,NaN,2026-01,City of London Police,City of London Police,-0.089208,51.509744,On or near Angel Lane,E01032739,City of London 001F,Anti-social behaviour,NaN
19435,NaN,2026-01,City of London Police,City of London Police,-0.107443,51.515534,On or near Printer Street,E01032740,City of London 001G,Anti-social behaviour,NaN


**For null values in crime_id, the records are retained because they represent valid crime incidents where an outcome had not yet been recorded.**

# **5. Convert Month to datetime**

In [12]:
city["month"] = pd.to_datetime(city["month"])
city.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17937 entries, 0 to 19525
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   crime_id               17480 non-null  object        
 1   month                  17937 non-null  datetime64[ns]
 2   reported_by            17937 non-null  object        
 3   falls_within           17937 non-null  object        
 4   longitude              17937 non-null  float64       
 5   latitude               17937 non-null  float64       
 6   location               17937 non-null  object        
 7   lsoa_code              17937 non-null  object        
 8   lsoa_name              17937 non-null  object        
 9   crime_type             17937 non-null  object        
 10  last_outcome_category  17480 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(8)
memory usage: 1.6+ MB


# **6. Remove duplicates**

In [13]:
# show rows with duplicates on crime_id
city[city.duplicated(subset=["crime_id"], keep=False)].sort_values(by="crime_id").head(10) # check the rows with duplicate crime_id to understand if there are any patterns in the duplicates

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category
2016,00b2af699e432e06687fc605eac627696f11e86a208ec4...,2024-03-01,City of London Police,City of London Police,-0.081902,51.517296,On or near Liverpool Street,E01032739,City of London 001F,Drugs,Status update unavailable
2018,00b2af699e432e06687fc605eac627696f11e86a208ec4...,2024-03-01,City of London Police,City of London Police,-0.081512,51.518692,On or near London Liverpool Street,E01032739,City of London 001F,Drugs,Status update unavailable
2223,039914afa967773aa97ca920049ee0f66f31906e734bb3...,2024-03-01,City of London Police,City of London Police,-0.104980,51.512033,On or near Kingscote Street,E01032740,City of London 001G,Violence and sexual offences,Status update unavailable
1917,039914afa967773aa97ca920049ee0f66f31906e734bb3...,2024-03-01,City of London Police,City of London Police,-0.104276,51.513712,On or near A201,E01032739,City of London 001F,Violence and sexual offences,Unable to prosecute suspect
7640,05dd7c247ed390f9bae36e78327d298b56e71735c9681e...,2024-10-01,City of London Police,City of London Police,-0.081902,51.517296,On or near Liverpool Street,E01032739,City of London 001F,Bicycle theft,Status update unavailable
7652,05dd7c247ed390f9bae36e78327d298b56e71735c9681e...,2024-10-01,City of London Police,City of London Police,-0.082133,51.515222,On or near St Helen'S Place,E01032739,City of London 001F,Bicycle theft,Status update unavailable
4877,062cb68b652fd6179f4d606c9db7a38fccc5a1ab94bb34...,2024-06-01,City of London Police,City of London Police,-0.102812,51.517051,On or near Police Station,E01032740,City of London 001G,Other theft,Court result unavailable
4874,062cb68b652fd6179f4d606c9db7a38fccc5a1ab94bb34...,2024-06-01,City of London Police,City of London Police,-0.106076,51.517239,On or near Conference/Exhibition Centre,E01032740,City of London 001G,Other theft,Status update unavailable
3949,073fe28119bb76ae7ec5526a695b167a3f4e8f892684a5...,2024-05-01,City of London Police,City of London Police,-0.101963,51.510761,On or near Ferry Terminal,E01032739,City of London 001F,Other crime,Status update unavailable
3948,073fe28119bb76ae7ec5526a695b167a3f4e8f892684a5...,2024-05-01,City of London Police,City of London Police,-0.079907,51.517785,On or near Victoria Avenue,E01032739,City of London 001F,Other crime,Court result unavailable


**Remove duplicates but exception of null values, as they would count as duplicates too.**

In [14]:
with_id = city[city["crime_id"].notnull()] # create a dataframe with rows that have a crime_id to check for duplicates
without_id = city[city["crime_id"].isnull()] # create a dataframe with rows that have no crime_id to check for duplicates

In [15]:
# remove duplicates with crime_id
with_id = with_id.sort_values(
    by = "last_outcome_category",
    key = lambda x: x == "Status update unavailable" # this pushes "Status update unavailable to the bottom" prioritising data with an actual outcome category
)
with_id = with_id.drop_duplicates(subset=["crime_id"], keep="first", inplace=False) # keep the first occurrence of each duplicate crime_id, which is the one with an actual outcome category
with_id.head()

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category
0,e7b720d0e1302d2d06db7b28b29132eb194864d44d7921...,2024-01-01,City of London Police,City of London Police,-0.106220,51.518275,On or near B500,E01000916,Camden 027B,Theft from the person,Investigation complete; no suspect identified
13030,2a33ee91d3b8587d89a4aadc0f1bddc6f899357841a76e...,2025-05-01,City of London Police,City of London Police,-0.095764,51.512549,On or near Friday Street,E01032739,City of London 001F,Drugs,Court result unavailable
13031,b9984981bea62f4be0ab00830a61b4d57f7d53ae9baed9...,2025-05-01,City of London Police,City of London Police,-0.079907,51.517785,On or near Victoria Avenue,E01032739,City of London 001F,Drugs,Court result unavailable
13032,4e5bbadc6ca10b1fee2d031cb32b5d0e074a4e687816b9...,2025-05-01,City of London Police,City of London Police,-0.081113,51.516869,On or near Bishopsgate,E01032739,City of London 001F,Drugs,Court result unavailable
13033,3f18609c4c0fc891da0a489a32901afd91bc9b93cd82bb...,2025-05-01,City of London Police,City of London Police,-0.101589,51.513515,On or near Ludgate Square,E01032739,City of London 001F,Other theft,Investigation complete; no suspect identified


**For crime_id with null values, assume when lsoa_code and crime_type are identical in a same month as duplicates.**

In [16]:
# show rows with duplicates on lsoa_code, crime_type and month
without_id[without_id.duplicated(subset=["lsoa_code", "crime_type", "month"], keep=False)].sort_values(by=["lsoa_code", "crime_type", "month"]).head()


,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category
10,NaN,2024-01-01,City of London Police,City of London Police,-0.097078,51.519045,On or near A1,E01000001,City of London 001A,Anti-social behaviour,NaN
11,NaN,2024-01-01,City of London Police,City of London Police,-0.097103,51.518461,On or near Aldersgate Street,E01000001,City of London 001A,Anti-social behaviour,NaN
12092,NaN,2025-04-01,City of London Police,City of London Police,-0.097078,51.519045,On or near A1,E01000001,City of London 001A,Anti-social behaviour,NaN
12093,NaN,2025-04-01,City of London Police,City of London Police,-0.097070,51.518892,On or near Aldersgate Street,E01000001,City of London 001A,Anti-social behaviour,NaN
761,NaN,2024-02-01,City of London Police,City of London Police,-0.095026,51.518499,On or near Park/Open Space,E01000002,City of London 001B,Anti-social behaviour,NaN


In [17]:
without_id = without_id.drop_duplicates(subset = ["lsoa_code", "crime_type", "month"], keep="first", inplace=False)
without_id.head()

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category
10,NaN,2024-01-01,City of London Police,City of London Police,-0.097078,51.519045,On or near A1,E01000001,City of London 001A,Anti-social behaviour,NaN
78,NaN,2024-01-01,City of London Police,City of London Police,-0.077511,51.515812,On or near Artizan Street,E01000005,City of London 001E,Anti-social behaviour,NaN
265,NaN,2024-01-01,City of London Police,City of London Police,-0.081113,51.516869,On or near Bishopsgate,E01032739,City of London 001F,Anti-social behaviour,NaN
575,NaN,2024-01-01,City of London Police,City of London Police,-0.101320,51.519293,On or near Grand Avenue,E01032740,City of London 001G,Anti-social behaviour,NaN
761,NaN,2024-02-01,City of London Police,City of London Police,-0.095026,51.518499,On or near Park/Open Space,E01000002,City of London 001B,Anti-social behaviour,NaN


In [18]:
# concat the dataframes with and without crime_id back together
city = pd.concat([with_id, without_id], ignore_index = True)

# **7. Categorical Columns**

**Categorising datas that have many repeated values as they reduce memory usage, and speeds up grouping operartions leading to improved performance.**

In [62]:
# Validating the data in crime_type and last_outcome_category are not duplicated ans is standardised
for crime in city["crime_type"].unique(): # check if 
    print(crime)
print("\n")
for crime in city["last_outcome_category"].unique(): # check if 
    print(crime)

Theft from the person
Drugs
Other theft
Burglary
Criminal damage and arson
Public order
Shoplifting
Vehicle crime
Violence and sexual offences
Other crime
Robbery
Bicycle theft
Possession of weapons
Anti-social behaviour


Investigation complete; no suspect identified
Court result unavailable
Unable to prosecute suspect
Further investigation is not in the public interest
Awaiting court outcome
Local resolution
Action to be taken by another organisation
Formal action is not in the public interest
Further action is not in the public interest
Offender given a caution
Offender given a drugs possession warning
Under investigation
Offender given penalty notice
Status update unavailable
nan


In [ ]:
# Check if lsoa_code corrctly corresponds to only one lsoa_name
check = city.groupby("lsoa_code", observed=True)["lsoa_name"].nunique() # check for unique name by grouping code
check[check > 1] # only prints if there are any unqiue names more than 1 

Series([], Name: lsoa_name, dtype: int64)

In [19]:
# show unique values in categorical columns: lsoa_code, lsoa_name, crime_type, last_outcome_category
print("Unique values in lsoa_code:", city["lsoa_code"].unique())
print("Unique values in lsoa_name:", city["lsoa_name"].unique())
print("Unique values in crime_type:", city["crime_type"].unique())
print("Unique values in last_outcome_category:", city["last_outcome_category"].unique())


Unique values in lsoa_code: ['E01000916' 'E01032739' 'E01000002' 'E01000003' 'E01000001' 'E01000005'
 'E01000914' 'E01000917' 'E01032740' 'E01001459' 'E01001499' 'E01033706'
 'E01033708' 'E01002724' 'E01003497' 'W01000574' 'W01000488' 'E01027857'
 'E01003929' 'E01003981' 'E01004307' 'E01001475' 'E01004293' 'E01004735'
 'E01000110' 'E01000013' 'E01004922' 'E01000967' 'E01032767' 'E01019412'
 'E01029407' 'E01002703' 'E01011227' 'E01028956' 'E01003934' 'E01032640'
 'E01029809' 'E01029812' 'E01004208' 'E01035678' 'E01004298' 'E01003756'
 'E01002783' 'E01001798' 'E01002704' 'E01033884' 'E01004322' 'E01004321'
 'E01035697' 'E01033490' 'E01003935' 'E01002905' 'E01004411' 'E01004286'
 'E01004734' 'E01002723' 'E01002711' 'E01003016' 'E01003489' 'E01003933'
 'E01004270' 'E01035687' 'E01004434' 'E01008940' 'E01000234' 'E01003013'
 'E01003108' 'E01001898' 'E01009650' 'E01001618' 'E01011226' 'E01027022'
 'E01000347' 'E01000731' 'E01010930' 'E01034792' 'E01001831' 'E01001755'
 'E01002793' 'E01002830

In [ ]:
# convert the categorical columns to category data type
cat_cols = [  # create list for looping
    "lsoa_name",
    "lsoa_code",
    "crime_type",
    "last_outcome_category"
]

for col in cat_cols:
    city[col] = city[col].astype("category")  # loop through each lists(columns) to convert it into category datatype

city.dtypes # check data types

crime_id                         object
month                    datetime64[ns]
reported_by                      object
falls_within                     object
longitude                       float64
latitude                        float64
location                         object
lsoa_code                      category
lsoa_name                      category
crime_type                     category
last_outcome_category          category
dtype: object

In [28]:
city.isnull().sum() # check the number of null values in each column to ensure there are no null values in the categorical columns

crime_id                 91
month                     0
reported_by               0
falls_within              0
longitude                 0
latitude                  0
location                  0
lsoa_code                 0
lsoa_name                 0
crime_type                0
last_outcome_category    91
dtype: int64

# **8. Seperate year and months to two different columns**

In [ ]:
city["year"] = city["month"].dt.year # seperate year into new column called year
city["month_num"] = city["month"].dt.month # seperate month number into new column called month_num
city.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17361 entries, 0 to 17360
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   crime_id               17270 non-null  object        
 1   month                  17361 non-null  datetime64[ns]
 2   reported_by            17361 non-null  object        
 3   falls_within           17361 non-null  object        
 4   longitude              17361 non-null  float64       
 5   latitude               17361 non-null  float64       
 6   location               17361 non-null  object        
 7   lsoa_code              17361 non-null  category      
 8   lsoa_name              17361 non-null  category      
 9   crime_type             17361 non-null  category      
 10  last_outcome_category  17270 non-null  category      
 11  year                   17361 non-null  int32         
 12  month_num              17361 non-null  int32         
dtypes

In [69]:
# checking shape
city.shape

(17361, 13)

In [ ]:
#city = pd.read_csv("data/2024-01/2024-01-city-of-london-street.csv")
#metro = pd.read_csv("data/2024-01/2024-01-metropolitan-street.csv")
#surrey = pd.read_csv("data/2024-01/2024-01-surrey-street.csv")
#thames = pd.read_csv("data/2024-01/2024-01-thames-valley-street.csv")
#wmids = pd.read_csv("data/2024-01/2024-01-west-midlands-street.csv")
#wyork = pd.read_csv("data/2024-01/2024-01-west-yorkshire-street.csv")

# **3.Metropolitan**

In [57]:
metro_files = sorted(glob.glob("../data/*/*metropolitan-street.csv")) # glob.glob returns a list of file paths matching the pattern, sorted to maintain order
metro = pd.concat([pd.read_csv(file) for file in metro_files], ignore_index=True) # ignore_index to reset the index after concatenation